# Simple MLRun Project

This notebook creates an MLRun project with a single `job` function that writes a message to the log.

In [ ]:
# %pip install mlrun==1.11.0

## 1. The function code

The job's code lives in [`hello_job.py`](hello_job.py), next to this notebook.

## 2. Create the project

Fill in [`mlrun.env`](mlrun.env) with your MLRun API URL (`MLRUN_DBPATH`), `V3IO_USERNAME` and `V3IO_ACCESS_KEY` before running this cell.

In [ ]:
import mlrun
import dotenv

dotenv.load_dotenv()

project = mlrun.get_or_create_project("hello-mlrun", context="./", user_project=True)
print(f"Project: {project.name}")

## 3. Use the git repo as the code source

With `pull_at_runtime=True`, every job run clones the repo at the given branch, so the job always runs the code committed to git.

In [ ]:
project.set_source("git://github.com/edmond-mck/safari-sagemaker.git#main", pull_at_runtime=True)

## 4. Register the job function

`with_repo=True` tells MLRun to load `hello_job.py` from the project source (the git repo) instead of embedding the local file.

In [ ]:
project.set_function(
    func="hello_job.py",
    name="hello",
    kind="job",
    image="mlrun/mlrun",
    handler="hello",
    with_repo=True,
)
project.save()

## 5. Run the job

The job runs on the MLRun cluster, where it clones the repo and runs `hello` from `hello_job.py`. The log message appears in the run's logs below and in the MLRun UI.

`local=True` would run the local copy of `hello_job.py` in this notebook instead of the code from git.

In [ ]:
run = project.run_function("hello", params={"name": "MLRun"})
run.outputs